# MACOG IaC-Eval Colab Runner

Runs the repo's paper-style IaC-Eval harness on Google Colab. Start with the v1 and v2 smoke tasks, then scale to full dataset runs.

Before running:
- Put model credentials in Colab Secrets or set them in the environment cell.
- Set `REPO_URL` to a Git repository containing the current MACOG changes.
- Runtime: CPU is fine; high-RAM helps for full runs.


In [ ]:
REPO_URL = "https://github.com/png261/MACOG-implement.git"
BRANCH = "main"
WORKDIR = "/content/strands-agent"

# Set this to True after connecting Colab to a local Jupyter runtime.
# In that mode, run Docker/MiniStack on your machine and expose it at 127.0.0.1:4566.
LOCAL_RUNTIME = False
EXTERNAL_AWS_ENDPOINT_URL = "http://127.0.0.1:4566"

V1_SMOKE_TASK = "451"
V2_SMOKE_TASK = "aws/task-057"
MODEL = "custom"
MAX_ITERATIONS = 1
TASK_TIMEOUT = 900

if LOCAL_RUNTIME:
    WORKDIR = "/tmp/strands-agent-colab-local"


In [ ]:
import os, pathlib, subprocess, textwrap

def run(cmd, cwd=None, env=None):
    print("$", cmd)
    return subprocess.run(cmd, shell=True, cwd=cwd, env=env or os.environ, check=True)

if LOCAL_RUNTIME:
    print("LOCAL_RUNTIME=True: skipping apt-get. Install terraform, opa, git, and python on the local machine.")
else:
    run("apt-get update -y")
    run("apt-get install -y unzip curl")

if not LOCAL_RUNTIME and not pathlib.Path('/usr/local/bin/terraform').exists():
    run("curl -fsSL -o /tmp/terraform.zip https://releases.hashicorp.com/terraform/1.13.5/terraform_1.13.5_linux_amd64.zip")
    run("unzip -o /tmp/terraform.zip -d /usr/local/bin")

if not LOCAL_RUNTIME and not pathlib.Path('/usr/local/bin/opa').exists():
    run("curl -fsSL -o /usr/local/bin/opa https://openpolicyagent.org/downloads/latest/opa_linux_amd64_static")
    run("chmod +x /usr/local/bin/opa")

run("terraform version")
run("opa version")


In [ ]:
import pathlib, shutil

if pathlib.Path(WORKDIR).exists():
    shutil.rmtree(WORKDIR)
run(f"git clone --branch {BRANCH} --depth 1 {REPO_URL} {WORKDIR}")
run("python -m pip install -U pip", cwd=WORKDIR)
run("python -m pip install -r requirements.txt", cwd=WORKDIR)


In [ ]:
import os, pathlib, shlex, shutil

try:
    from google.colab import userdata
    for key in ["CUSTOM_API_KEY", "CUSTOM_BASE_URL", "CUSTOM_MODEL_ID", "OPENROUTER_API_KEY", "OPENROUTER_MODEL"]:
        value = userdata.get(key)
        if value:
            os.environ[key] = value
except Exception:
    pass

DEPLOY_MODE = "sandbox" if LOCAL_RUNTIME else "structural"
os.environ.setdefault("MACOG_MODEL_BACKEND", "custom")
os.environ["MACOG_DEVOPS_MODE"] = DEPLOY_MODE
if LOCAL_RUNTIME:
    os.environ["AWS_ENDPOINT_URL"] = EXTERNAL_AWS_ENDPOINT_URL
    os.environ.pop("MACOG_EVAL_FAST_SCHEMA", None)
else:
    os.environ.setdefault("MACOG_EVAL_FAST_SCHEMA", "1")
if LOCAL_RUNTIME:
    os.environ.setdefault("OPA_BIN", shutil.which("opa") or "opa")
else:
    os.environ.setdefault("OPA_BIN", "/usr/local/bin/opa")
os.environ.setdefault("TF_INPUT", "0")
os.environ.setdefault("TF_IN_AUTOMATION", "1")
os.environ.setdefault("AWS_DEFAULT_REGION", "us-east-1")

env_keys = [
    "CUSTOM_API_KEY", "CUSTOM_BASE_URL", "CUSTOM_MODEL_ID",
    "OPENROUTER_API_KEY", "OPENROUTER_MODEL", "ANTHROPIC_API_KEY", "CLAUDE_MODEL_ID",
    "MACOG_MODEL_BACKEND", "MACOG_DEVOPS_MODE", "MACOG_EVAL_FAST_SCHEMA",
    "AWS_ENDPOINT_URL", "AWS_DEFAULT_REGION", "OPA_BIN", "TF_INPUT", "TF_IN_AUTOMATION",
]
env_path = pathlib.Path(WORKDIR) / ".env"
env_lines = []
for key in env_keys:
    value = os.environ.get(key)
    if value:
        env_lines.append(f"{key}={shlex.quote(value)}")
env_path.write_text("\n".join(env_lines) + "\n")
env_path.chmod(0o600)
print(f"Deploy mode: {DEPLOY_MODE}; AWS_ENDPOINT_URL={os.environ.get('AWS_ENDPOINT_URL', '(none)')}")
print(f"Wrote {env_path} with {len(env_lines)} keys; secret values are not printed.")

required = ["CUSTOM_API_KEY", "CUSTOM_BASE_URL", "CUSTOM_MODEL_ID"] if MODEL == "custom" else []
missing = [k for k in required if not os.environ.get(k)]
if missing:
    raise RuntimeError(f"Missing Colab secrets/env vars: {missing}")


In [ ]:
run("python -m py_compile eval/dataset.py eval/run_eval.py eval/_worker.py eval/metrics.py eval/models.py agents/reviewer/tools.py", cwd=WORKDIR)


## Preflight Test First\n\nRun this before any model call. It verifies that the eval package imports, both datasets load, OPA is callable, and the local MiniStack endpoint is reachable when `LOCAL_RUNTIME=True`.\n

In [ ]:
import json, shutil, subprocess\n\ndef capture(cmd, cwd=WORKDIR):\n    print("$", cmd)\n    out = subprocess.check_output(cmd, shell=True, cwd=cwd, text=True, stderr=subprocess.STDOUT)\n    print(out[:2000])\n    return out\n\nassert shutil.which("terraform"), "terraform is not on PATH"\nassert shutil.which("opa") or os.environ.get("OPA_BIN"), "opa is not available"\ncapture("terraform version")\ncapture(f"{os.environ.get('OPA_BIN', 'opa')} version")\nif LOCAL_RUNTIME:\n    capture(f"python - <<'PY'\nimport os, requests\nurl = os.environ['AWS_ENDPOINT_URL'].rstrip('/') + '/_ministack/health'\nr = requests.get(url, timeout=5)\nprint(url, r.status_code, r.text[:300])\nr.raise_for_status()\nPY")\n\ntest_code = r'''\nfrom eval.dataset import load_tasks\nv1 = load_tasks(dataset="iac-eval-v1", config="default", task_ids=["451"], limit=1)\nv2 = load_tasks(dataset="iac-eval-v2", config="default", task_ids=["aws/task-057"], limit=1)\nassert len(v1) == 1, "v1 smoke task did not load"\nassert len(v2) == 1, "v2 smoke task did not load"\nassert v1[0]["rego_intent"], "v1 Rego policy missing"\nassert v2[0]["rego_intent"], "v2 Rego policy missing"\nprint({"v1": {"id": v1[0]["id"], "difficulty": v1[0]["difficulty"], "resources": v1[0]["resources"]}, "v2": {"id": v2[0]["id"], "difficulty": v2[0]["difficulty"], "resources": v2[0]["resources"]}})\n'''\ncapture("python - <<\'PY\'\n" + test_code + "\nPY")\nprint("Preflight passed. Now run the v1 smoke cell.")\n

## Smoke: IaC-Eval v1


In [ ]:
run(
    f"python -m eval.run_eval --dataset iac-eval-v1 --models {MODEL} --config default "
    f"--task-id {V1_SMOKE_TASK} --limit 1 --max-iterations {MAX_ITERATIONS} "
    f"--deploy-mode {DEPLOY_MODE} --task-timeout {TASK_TIMEOUT} "
    "--output eval/results/colab_iac_eval_v1_smoke",
    cwd=WORKDIR,
)


## Smoke: IaC-Eval v2


In [ ]:
run(
    f"python -m eval.run_eval --dataset iac-eval-v2 --models {MODEL} --config default "
    f"--task-id {V2_SMOKE_TASK} --limit 1 --max-iterations {MAX_ITERATIONS} "
    f"--deploy-mode {DEPLOY_MODE} --task-timeout {TASK_TIMEOUT} "
    "--output eval/results/colab_iac_eval_v2_smoke",
    cwd=WORKDIR,
)


## Full Runs

Run these only after both smoke tasks pass. Increase `MAX_ITERATIONS` to match the paper setting you want to compare.


In [ ]:
# Full v1: 458 tasks
# run(
#     f"python -m eval.run_eval --dataset iac-eval-v1 --models {MODEL} --config default "
#     f"--max-iterations 3 --deploy-mode {DEPLOY_MODE} --task-timeout 1200 "
#     "--output eval/results/colab_iac_eval_v1_full",
#     cwd=WORKDIR,
# )

# Full v2: 186 tasks
# run(
#     f"python -m eval.run_eval --dataset iac-eval-v2 --models {MODEL} --config default "
#     f"--max-iterations 3 --deploy-mode {DEPLOY_MODE} --task-timeout 1200 "
#     "--output eval/results/colab_iac_eval_v2_full",
#     cwd=WORKDIR,
# )


In [ ]:
run("find eval/results -name 'paper_summary_*.md' -print -maxdepth 4", cwd=WORKDIR)
